# Dự báo chuỗi thời gian EV Total bằng SARIMA (Mục 4.5)

- Dữ liệu: `Electric_Vehicle_Population_Data.csv` (dataset [H]), tổng EV toàn bang Washington theo tháng.
- **Train:** 71 tháng (01/2017 – 11/2022). **Test:** 12 tháng (12/2022 – 11/2023).
- Cấu hình chỉ chọn dựa trên Train (ADF, ACF/PACF, AIC/BIC, phần dư); Test chỉ dùng để đánh giá.
  - **A:** SARIMA(1,1,1)×(1,1,0)₁₂
  - **B:** SARIMA(0,1,1)×(1,1,1)₁₂
  - **C:** SARIMA(0,2,1)×(0,1,1)₁₂ (d = 2 theo ADF)

## 1. Thư viện & đường dẫn (tự nhận diện Colab / local)

In [ ]:
%matplotlib inline
import json, sys, io, warnings
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from IPython.display import display, Image
from scipy import stats
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import adfuller, acf, pacf
from statsmodels.stats.diagnostic import acorr_ljungbox

warnings.filterwarnings("ignore")
plt.rcParams["font.sans-serif"] = ["DejaVu Sans", "Arial", "sans-serif"]
plt.rcParams["axes.unicode_minus"] = False

def show(fig):
    # Luôn render PNG và hiển thị ngay dưới cell (hoạt động với mọi backend/kernel), rồi đóng figure
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=110, bbox_inches="tight", facecolor="white")
    display(Image(data=buf.getvalue()))
    plt.close(fig)

CSV_NAME = "Electric_Vehicle_Population_Data.csv"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # Kernel Colab không thấy file trong repo local -> tìm trong /content, nếu chưa có thì upload
    DATA_PATH = Path("/content") / CSV_NAME
    if not DATA_PATH.exists():
        from google.colab import files
        print(f"Hãy chọn file {CSV_NAME} để upload...")
        files.upload()
    OUT_DIR = Path("/content/sarima_out")
else:
    candidates = [Path("../data") / CSV_NAME, Path("data") / CSV_NAME]
    DATA_PATH = next((p for p in candidates if p.exists()), candidates[0])
    OUT_DIR = (Path("../results") if Path("../results").exists() else Path("results")) / "sarima_out"

OUT_DIR.mkdir(parents=True, exist_ok=True)
assert DATA_PATH.exists(), f"Không tìm thấy dataset: {DATA_PATH}"
print("Môi trường:", "Colab" if IN_COLAB else "Local")
print("Dataset  :", DATA_PATH.resolve())
print("Kết quả  :", OUT_DIR.resolve())

## 2. Tải dữ liệu & chia Train/Test

In [ ]:
N_TRAIN, N_TEST, S = 71, 12, 12

def load_ev_series(path):
    H = pd.read_csv(path)
    H.columns = ["Date", "County", "State", "Use", "BEV", "PHEV", "EV", "NonEV", "Total", "Pct"]
    H["Date"] = pd.to_datetime(H["Date"])
    y = H.groupby("Date")["EV"].sum().astype(float)
    y.index = pd.DatetimeIndex(y.index).to_period("M").to_timestamp("M")
    y = y.asfreq("ME")
    y.name = "EV_Total"
    return y

y = load_ev_series(DATA_PATH)
train, test = y.iloc[:N_TRAIN], y.iloc[N_TRAIN:]
assert len(train) == N_TRAIN and len(test) == N_TEST

print(f"Tổng: {len(y)} tháng ({y.index.min():%m/%Y} -> {y.index.max():%m/%Y})")
print(f"Train: {len(train)} tháng | Test: {len(test)} tháng")
display(y.to_frame().iloc[list(range(3)) + list(range(-3, 0))])

## 3. Biểu đồ chuỗi thời gian

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(train.index, train.values, color="#1f77b4", lw=2, label="Train (71 tháng)")
ax.plot(test.index, test.values, color="#ff7f0e", lw=2, ls="--", label="Test (12 tháng)")
ax.axvline(train.index[-1], color="grey", ls=":", lw=1.2)
ax.set_title("EV Total – Washington (2017–2023)", fontweight="bold")
ax.set_xlabel("Thời gian"); ax.set_ylabel("Số xe")
ax.legend()
fig.tight_layout(); fig.savefig(OUT_DIR / "hinh_chuoi_thoi_gian.png", dpi=180, facecolor="white")
show(fig)

## 4. Kiểm định ADF (chỉ trên Train)

In [ ]:
def band(n): return 1.96 / np.sqrt(n)

series = {
    "d=1": train.diff(),
    "d=1,D=1": train.diff().diff(S),
    "d=2": train.diff().diff(),
    "d=2,D=1": train.diff().diff().diff(S),
}
rows = []
for name, s in {"level": train, **series}.items():
    x = s.dropna()
    stat, p = adfuller(x, regression="c")[:2]
    rows.append({"Chuỗi": name, "n": len(x), "ADF stat": round(stat, 4), "p-value": round(p, 4),
                 "Kết luận": "Dừng" if p < 0.05 else "Chưa dừng"})
display(pd.DataFrame(rows))

## 5. ACF / PACF sau sai phân

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 7))
for r_, k in enumerate(["d=1,D=1", "d=2,D=1"]):
    x = series[k].dropna(); n = len(x); b = band(n)
    a = acf(x, nlags=36); p = pacf(x, nlags=min(24, n // 2 - 1), method="ywm")
    for c_, (vals, nm) in enumerate([(a, "ACF"), (p, "PACF")]):
        ax = axes[r_, c_]
        ax.bar(range(1, len(vals)), vals[1:], width=0.5, color="#2F5597")
        ax.axhline(b, color="#C55A11", ls="--", lw=1); ax.axhline(-b, color="#C55A11", ls="--", lw=1)
        ax.axhline(0, color="k", lw=.6)
        for l in (12, 24, 36): ax.axvline(l, color="grey", ls=":", lw=1)
        ax.set_title(f"{nm} sau sai phân {k} (n = {n})", fontsize=10.5, fontweight="bold")
        ax.set_xlabel("Độ trễ (tháng)"); ax.set_ylim(-.8, 1.0 if c_ == 0 else .8)
fig.tight_layout(); fig.savefig(OUT_DIR / "hinh_acf_pacf.png", dpi=180, facecolor="white")
show(fig)

## 6. Ước lượng 3 cấu hình SARIMA

In [ ]:
CONFIGS = {
    "A": ((1, 1, 1), (1, 1, 0, S)),
    "B": ((0, 1, 1), (1, 1, 1, S)),
    "C": ((0, 2, 1), (0, 1, 1, S)),
}
fits, resids, rows = {}, {}, []
for key, (o, so) in CONFIGS.items():
    m = SARIMAX(train, order=o, seasonal_order=so, trend="n",
                enforce_stationarity=True, enforce_invertibility=True).fit(disp=False, maxiter=500)
    burn = o[1] + so[1] * S
    r = m.resid.iloc[burn:]
    mdf = o[0] + o[2] + so[0] + so[2]
    lb = acorr_ljungbox(r, lags=[12, 24], model_df=mdf)
    fits[key], resids[key] = m, (r, burn)
    rows.append({"Cấu hình": key, "Order": str(o), "Seasonal": str(so),
                 "AIC": round(m.aic, 2), "BIC": round(m.bic, 2),
                 "LB p(12)": round(lb.lb_pvalue.iloc[0], 4), "LB p(24)": round(lb.lb_pvalue.iloc[1], 4),
                 "JB p": round(stats.jarque_bera(r)[1], 4),
                 "Hội tụ": bool(m.mle_retvals.get("converged"))})
df_train = pd.DataFrame(rows)
display(df_train)

## 7. Chẩn đoán phần dư

In [ ]:
fig, ax = plt.subplots(3, 2, figsize=(11, 8.5))
for i, k in enumerate("ABC"):
    r, burn = resids[k]
    a = acf(r, nlags=24); b = band(len(r))
    ax[i, 0].plot(r.index, r.values, color="#2F5597", lw=1.3); ax[i, 0].axhline(0, color="k", lw=.6)
    ax[i, 0].set_title(f"Phần dư cấu hình {k} (bỏ {burn} giá trị đầu)", fontsize=10.5, fontweight="bold")
    ax[i, 1].bar(range(1, 25), a[1:], width=.5, color="#2F5597")
    ax[i, 1].axhline(b, color="#C55A11", ls="--", lw=1); ax[i, 1].axhline(-b, color="#C55A11", ls="--", lw=1)
    ax[i, 1].axvline(12, color="grey", ls=":", lw=1)
    ax[i, 1].set_title(f"ACF phần dư cấu hình {k}", fontsize=10.5, fontweight="bold"); ax[i, 1].set_ylim(-.6, .8)
fig.tight_layout(); fig.savefig(OUT_DIR / "hinh_phan_du.png", dpi=180, facecolor="white")
show(fig)

## 8. Dự báo 12 tháng Test & đánh giá

In [ ]:
def metrics(actual, fc):
    e = actual - fc
    return {"MAE": np.mean(abs(e)), "RMSE": np.sqrt(np.mean(e ** 2)),
            "MAPE (%)": np.mean(abs(e / actual)) * 100,
            "sMAPE (%)": np.mean(2 * abs(e) / (abs(actual) + abs(fc))) * 100,
            "Bias": np.mean(e)}

fc_tab = pd.DataFrame({"date": test.index.strftime("%Y-%m"), "actual": test.values})
fc_cache, rows = {}, []
for k in "ABC":
    f = fits[k].get_forecast(N_TEST)
    fm, ci = f.predicted_mean, f.conf_int(alpha=0.05)
    fc_cache[k] = (fm, ci)
    mt = metrics(test.values, fm.values)
    cover = int(((test.values >= ci.iloc[:, 0].values) & (test.values <= ci.iloc[:, 1].values)).sum())
    fc_tab[f"{k}_forecast"] = fm.values
    fc_tab[f"{k}_lo95"] = ci.iloc[:, 0].values
    fc_tab[f"{k}_hi95"] = ci.iloc[:, 1].values
    rows.append({"Cấu hình": k, "AIC": round(fits[k].aic, 2), "BIC": round(fits[k].bic, 2),
                 **{n: round(v, 2) for n, v in mt.items()}, "Phủ KDB 95%": f"{cover}/12"})
df_test = pd.DataFrame(rows)
display(df_test)
display(fc_tab.round(1))

## 9. Biểu đồ dự báo & lưu kết quả

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(14, 4.2), sharey=True)
tr = train.iloc[-24:]
for i, k in enumerate("ABC"):
    fm, ci = fc_cache[k]
    ax[i].plot(tr.index, tr.values, color="grey", lw=1.5, label="Train (24 tháng cuối)")
    ax[i].plot(test.index, test.values, color="k", lw=2, label="Thực tế")
    ax[i].plot(test.index, fm.values, color="#C55A11", lw=2, ls="--", label="Dự báo")
    ax[i].fill_between(test.index, ci.iloc[:, 0], ci.iloc[:, 1], color="#C55A11", alpha=.18, label="Khoảng dự báo 95%")
    mape = df_test.loc[df_test["Cấu hình"] == k, "MAPE (%)"].iloc[0]
    ax[i].set_title(f"Cấu hình {k} (MAPE {mape}%)", fontweight="bold")
    ax[i].tick_params(axis="x", rotation=45, labelsize=8.5)
ax[0].legend(fontsize=8.5); ax[0].set_ylabel("EV Total (xe)")
fig.tight_layout(); fig.savefig(OUT_DIR / "hinh_du_bao.png", dpi=180, facecolor="white")
show(fig)

fc_tab.round(1).to_csv(OUT_DIR / "sarima_forecasts_test.csv", index=False)
df_test.to_csv(OUT_DIR / "sarima_summary.csv", index=False)
res = {k: {"order": CONFIGS[k][0], "seasonal_order": CONFIGS[k][1], "aic": float(fits[k].aic),
           "bic": float(fits[k].bic), "params": {n: float(v) for n, v in fits[k].params.items()},
           "pvalues": {n: float(v) for n, v in fits[k].pvalues.items()}} for k in "ABC"}
(OUT_DIR / "sarima_results.json").write_text(json.dumps(res, indent=1), encoding="utf-8")
print("Đã lưu kết quả vào:", OUT_DIR.resolve())